# VPA Demo: LLM Agent

This notebook demonstrates the agent-facing command flow without requiring live
LLM credentials by default. It inspects the local VPA surface and prints the
plan-only and execution commands. Set `VPA_DEMO_RUN_LLM=1` to make live provider
calls.

In [ ]:
from __future__ import annotations

import json
import os
import shlex
import shutil
import subprocess
from pathlib import Path


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for path in [here, *here.parents]:
        if (path / "pyproject.toml").is_file() and (path / "src" / "vibeproteinagent").is_dir():
            return path
    return here


REPO_ROOT = find_repo_root()
RUN_INSTALL = os.environ.get("VPA_DEMO_RUN_INSTALL", "0") == "1"
RUN_HEAVY = os.environ.get("VPA_DEMO_RUN_HEAVY", "0") == "1"
RUN_LLM = os.environ.get("VPA_DEMO_RUN_LLM", "0") == "1"
WORKSPACE_PREFIX = os.environ.get("VPA_DEMO_WORKSPACE_PREFIX", f"vpa_demo_{os.getpid()}")
OUT_DIR = Path(os.environ.get("VPA_DEMO_OUT_DIR", "/tmp")) / WORKSPACE_PREFIX
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Keep the demo daemon/socket isolated from any VPA daemon the user may already be running.
os.environ.setdefault("VPA_RUNTIME_DIR", str(OUT_DIR / "runtime"))
Path(os.environ["VPA_RUNTIME_DIR"]).mkdir(parents=True, exist_ok=True)


def _candidate_vpa_commands() -> list[list[str]]:
    commands: list[list[str]] = []
    installed = shutil.which("vpa")
    if installed:
        commands.append([installed])
    if (REPO_ROOT / "pyproject.toml").is_file() and shutil.which("uv"):
        commands.append(["uv", "run", "vpa"])
    return commands


def choose_vpa() -> list[str]:
    errors = []
    for base in _candidate_vpa_commands():
        proc = subprocess.run(
            [*base, "version", "--json"],
            cwd=REPO_ROOT,
            text=True,
            capture_output=True,
            env=os.environ.copy(),
            timeout=120,
        )
        if proc.returncode == 0:
            print("Using VPA command:", " ".join(shlex.quote(part) for part in base))
            print(proc.stdout.strip())
            return base
        errors.append((base, proc.returncode, proc.stderr.strip() or proc.stdout.strip()))
    raise RuntimeError(f"Could not find a working vpa command: {errors}")


VPA = choose_vpa()


def show_command(parts: list[str]) -> None:
    print("$ " + " ".join(shlex.quote(str(part)) for part in parts))


def run(parts: list[str], *, check: bool = True, timeout: int = 300, cwd: Path | None = None) -> subprocess.CompletedProcess[str]:
    parts = [str(part) for part in parts]
    show_command(parts)
    proc = subprocess.run(
        parts,
        cwd=cwd or REPO_ROOT,
        text=True,
        capture_output=True,
        env=os.environ.copy(),
        timeout=timeout,
    )
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print(proc.stderr)
    if check and proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, parts, proc.stdout, proc.stderr)
    return proc


def run_json(parts: list[str], *, timeout: int = 300, check: bool = True) -> dict:
    proc = run([*parts, "--json"], timeout=timeout, check=check)
    if proc.returncode != 0:
        return {}
    return json.loads(proc.stdout)


def stop_demo_daemon() -> None:
    run([*VPA, "daemon", "stop", "--workspace", f"{WORKSPACE_PREFIX}_cleanup", "--json"], check=False, timeout=60)


print("Repo root:", REPO_ROOT)
print("Output directory:", OUT_DIR)
print("Runtime directory:", os.environ["VPA_RUNTIME_DIR"])
print("RUN_INSTALL=", RUN_INSTALL, "RUN_HEAVY=", RUN_HEAVY, "RUN_LLM=", RUN_LLM)

## Optional Agent Install

In [ ]:
install_cmd = ["npx", "vibeproteinagent", "install", "--profile", "agent", "--cuda-profile", "auto", "--yes"]
if RUN_INSTALL:
    run(install_cmd, timeout=3600)
else:
    print("Skipping install. To run it, set VPA_DEMO_RUN_INSTALL=1.")
    show_command(install_cmd)

## 1. Local Agent Readiness Checks

These checks do not call an LLM provider.

In [ ]:
run([*VPA, "agent", "run", "--help"], timeout=120)
run_json([*VPA, "tools"], timeout=120)
run_json([*VPA, "workflow", "list"], timeout=120)

## 2. Provider Setup Commands

In [ ]:
print("Provider setup examples:")
show_command(["codex", "login"])
show_command(["claude", "/login"])
print("export OPENROUTER_API_KEY=<openrouter-api-key>")
print("Current provider:", os.environ.get("VPA_DEMO_PROVIDER", "codex"))
print("Current model:", os.environ.get("VPA_DEMO_MODEL", "gpt-5.2"))

## 3. Plan-Only Request

`--plan-only` still calls the selected provider, so the notebook only executes
it when `VPA_DEMO_RUN_LLM=1`.

In [ ]:
workspace = f"{WORKSPACE_PREFIX}_agent"
request = "Analyze this short protein sequence with VPA tools, run QC, rank any candidate records, and report the result."
provider = os.environ.get("VPA_DEMO_PROVIDER", "codex")
model = os.environ.get("VPA_DEMO_MODEL", "gpt-5.2")
plan_cmd = [
    *VPA,
    "agent",
    "run",
    request,
    "--provider",
    provider,
    "--model",
    model,
    "--workspace",
    workspace,
    "--param",
    "sequence=ACDEFGHIKLMNPQRSTVWY",
    "--plan-only",
]
if RUN_LLM:
    run(plan_cmd, timeout=900)
else:
    print("Skipping live plan-only call. To execute it, set VPA_DEMO_RUN_LLM=1 and configure the provider.")
    show_command(plan_cmd)

## 4. Execute The Agent

Execution needs a provider plus workers. The command is printed by default.

In [ ]:
execute_cmds = [
    [*VPA, "daemon", "start", "--cpu-workers", "1", "--workspace", workspace],
    [
        *VPA,
        "agent",
        "run",
        request,
        "--provider",
        provider,
        "--model",
        model,
        "--workspace",
        workspace,
        "--param",
        "sequence=ACDEFGHIKLMNPQRSTVWY",
        "--max-rounds",
        "4",
        "--max-entries",
        "3",
        "--max-wait-s",
        "3600",
    ],
]
if RUN_LLM:
    stop_demo_daemon()
    for cmd in execute_cmds:
        run(cmd, timeout=3600)
else:
    print("Skipping live agent execution. To execute it, set VPA_DEMO_RUN_LLM=1.")
    for cmd in execute_cmds:
        show_command(cmd)

## 5. Inspect Runs And Generate Reports

In [ ]:
inspect_cmds = [
    [*VPA, "agent", "runs", "--workspace", workspace],
    [*VPA, "agent", "show", "<agent_run_id>", "--workspace", workspace],
    [*VPA, "workflow", "status", "<workflow_id>", "--workspace", workspace],
    [*VPA, "report", "<workflow_id>", "--workspace", workspace, "--out", str(OUT_DIR / "agent_report.html")],
]
if RUN_LLM:
    run(inspect_cmds[0], check=False, timeout=120)
else:
    print("Commands to inspect a completed agent run:")
    for cmd in inspect_cmds:
        show_command(cmd)

## 6. Agent Workflow Skills

The public user docs explain how to run VPA. Agent operating protocol lives in
`skills.md` and `docs/agent_workflows/`; those files tell Codex, Claude Code,
and similar agents how to preserve candidate archive, compiler, decision, and
reporting contracts during executable protein workflows.

In [ ]:
for path in [
    REPO_ROOT / "skills.md",
    REPO_ROOT / "docs" / "agent_workflows" / "protein_design_skill.md",
    REPO_ROOT / "docs" / "agent_workflows" / "vpa_contracts.md",
    REPO_ROOT / "docs" / "agent_workflows" / "live_matrix_validation.md",
]:
    print(path.relative_to(REPO_ROOT), "exists=", path.is_file())

## 7. Stop The Demo Daemon If One Was Started

In [ ]:
if RUN_LLM:
    run([*VPA, "daemon", "stop", "--workspace", workspace, "--json"], check=False, timeout=60)
else:
    print("No daemon was started in default no-LLM mode.")